<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 9 — Управление заказами


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Продолжение проекта Sprint3/Task3: онлайн-заказы, физические, специализированные и экспресс-заказы. Добавлены данные об отзывах, счетах, упаковке, согласовании условий и вызове курьера. Реализованы явные методы интерфейсов и сервис уведомлений с передачей зависимости через конструктор.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;

interface ITrackable
{
    string TrackingNumber { get; set; }
    void Track();
}

interface INotifiable
{
    void NotifyCustomer();
}

class Item
{
    private decimal price = 0m;
    private int quantity = 1;

    public string Name { get; }

    public decimal Price
    {
        get { return price; }
        set
        {
            if (value < 0)
                throw new ArgumentOutOfRangeException("price", "Цена не может быть отрицательной.");

            price = value;
        }
    }

    public int Quantity
    {
        get { return quantity; }
        set
        {
            if (value <= 0)
                throw new ArgumentOutOfRangeException("quantity", "Количество должно быть положительным.");

            quantity = value;
        }
    }

    public Item(string name, decimal price = 0m, int quantity = 1)
    {
        if (string.IsNullOrWhiteSpace(name))
            throw new ArgumentException("Название позиции не должно быть пустым.", "name");

        Name = name;
        Price = price;
        Quantity = quantity;
    }

    public decimal GetCost() { return Price * Quantity; }
}

abstract class Order
{

    public string CustomerCity { get; private set; } = "Не указан";
    public int ReviewRating { get; private set; }
    public string ReviewText { get; private set; } = "Нет";

    public void SetCity(string city) { CustomerCity = city; }

    public void LeaveReview(int rating, string text)
    {
        if (rating < 1 || rating > 5)
            throw new ArgumentOutOfRangeException("rating", "Оценка должна быть от 1 до 5.");
        ReviewRating = rating;
        ReviewText = text;
    }

    public void ShowReview()
    {
        Console.WriteLine($"Город: {CustomerCity}, оценка: {ReviewRating}, отзыв: {ReviewText}.");
    }

    private List<Item> items = new List<Item>();

    public int OrderId { get; }
    public DateTime CreationDate { get; }
    public decimal TotalAmount { get; protected set; }
    public int ItemCount { get { return items.Count; } }

    public string CustomerName { get; set; } = "Не указан";
    public string Status { get; protected set; } = "Новый";
    public string Comment { get; set; } = "Нет";

    public string CustomerPhone { get; private set; } = "Не указан";
    public bool IsUrgent { get; private set; } = false;
    public bool IsArchived { get; private set; } = false;

    protected Order(int orderId, DateTime creationDate)
    {
        if (orderId <= 0)
            throw new ArgumentOutOfRangeException("orderId", "Номер заказа должен быть положительным.");

        OrderId = orderId;
        CreationDate = creationDate;
    }

    public void ConfirmOrder() { Status = "Подтверждён"; }
    public void CancelOrder() { Status = "Отменён"; }

    public void ConfirmOrder(string customerPhone, bool isUrgent)
    {
        CustomerPhone = customerPhone;
        IsUrgent = isUrgent;
        ConfirmOrder();
    }

    public void Archive() { IsArchived = true; }

    public void AddItem(string name, decimal price, int quantity = 1)
    {
        AddItem(new Item(name, price, quantity));
    }

    public virtual void ShowOrderInfo()
    {
        CalculateTotal();
        Console.WriteLine($"Заказ #{OrderId}, дата {CreationDate:dd.MM.yyyy}, позиций: {ItemCount}, сумма: {TotalAmount:F2} руб.");
        ShowItems();
        Console.WriteLine($"Клиент: {CustomerName}, статус: {Status}, комментарий: {Comment}.");
        Console.WriteLine($"Телефон: {CustomerPhone}, срочный: {IsUrgent}, в архиве: {IsArchived}.");
    }

    public virtual decimal CalculateTotal()
    {
        TotalAmount = 0;
        foreach (Item item in items)
            TotalAmount += item.GetCost();

        return TotalAmount;
    }

    public virtual void AddItem(Item item)
    {
        if (item == null)
            throw new ArgumentNullException("item");

        items.Add(item);
        CalculateTotal();
    }

    public virtual bool RemoveItem(Item item)
    {
        if (item == null)
            throw new ArgumentNullException("item");

        bool removed = items.Remove(item);
        if (removed)
            CalculateTotal();

        return removed;
    }

    public bool TransferItemTo(Order targetOrder, Item item)
    {
        if (targetOrder == null)
            throw new ArgumentNullException("targetOrder");

        if (!RemoveItem(item))
            return false;

        targetOrder.AddItem(item);
        return true;
    }

    public void ShowItems()
    {
        foreach (Item item in items)
            Console.WriteLine($"  {item.Name}: {item.Price:F2} руб. x {item.Quantity} = {item.GetCost():F2} руб.");
    }
}

class OnlineOrder : Order, ITrackable, INotifiable
{

    public string InvoiceNumber { get; private set; } = "Нет";
    public string InvoiceEmail { get; private set; } = "Не указан";
    public bool IsInvoiceSent { get; private set; }

    public void CreateInvoice(string number, string email)
    {
        if (string.IsNullOrWhiteSpace(number) || string.IsNullOrWhiteSpace(email))
            throw new ArgumentException("Укажите номер счёта и email.");
        InvoiceNumber = number;
        InvoiceEmail = email;
        IsInvoiceSent = false;
    }

    public void SendInvoice()
    {
        if (InvoiceNumber == "Нет")
        {
            Console.WriteLine("Сначала создайте счёт.");
            return;
        }
        IsInvoiceSent = true;
    }

    public void ShowInvoice()
    {
        Console.WriteLine($"Счёт: {InvoiceNumber}, email: {InvoiceEmail}, отправлен: {IsInvoiceSent}.");
    }

    void ITrackable.Track() { Track(); }
    void INotifiable.NotifyCustomer() { NotifyCustomer(); }

    public string CustomerEmail { get; }
    public string DeliveryMethod { get; }

    public string Website { get; set; } = "shop.example";
    public string TrackingNumber { get; set; } = "Не присвоен";
    public bool IsPaid { get; private set; } = false;

    public string PaymentMethod { get; private set; } = "Не указан";
    public string ReceiptNumber { get; private set; } = "Нет";
    public bool IsReceiptCreated { get; private set; } = false;

    public OnlineOrder(int orderId, DateTime creationDate,
                       string customerEmail, string deliveryMethod)
        : base(orderId, creationDate)
    {
        if (string.IsNullOrWhiteSpace(customerEmail))
            throw new ArgumentException("Email не должен быть пустым.", "customerEmail");
        if (string.IsNullOrWhiteSpace(deliveryMethod))
            throw new ArgumentException("Укажите способ доставки.", "deliveryMethod");

        CustomerEmail = customerEmail;
        DeliveryMethod = deliveryMethod;
    }

    public override void AddItem(Item item)
    {
        base.AddItem(item);
        Console.WriteLine($"Онлайн-заказ #{OrderId}: добавлено «{item.Name}»; доставка — {DeliveryMethod}.");
    }

    public void ShowDeliveryInfo()
    {
        Console.WriteLine($"Заказ #{OrderId}: {DeliveryMethod}; email клиента: {CustomerEmail}.");
    }

    public void MarkAsPaid()
    {
        CalculateTotal();
        IsPaid = true;
        Console.WriteLine($"Заказ #{OrderId}: оплата отмечена: {IsPaid}, сумма: {TotalAmount:F2} руб.");
    }

    public void Track()
    {
        Console.WriteLine($"Отслеживание #{OrderId}: сайт {Website}, номер {TrackingNumber}.");
    }

    public void NotifyCustomer()
    {
        Console.WriteLine($"Уведомление для {CustomerEmail}: заказ #{OrderId}, статус: {Status}.");
    }

    public void MarkAsPaid(string paymentMethod)
    {
        if (string.IsNullOrWhiteSpace(paymentMethod))
            throw new ArgumentException("Укажите способ оплаты.", "paymentMethod");

        PaymentMethod = paymentMethod;
        MarkAsPaid();
    }

    public void CreateReceipt(string receiptNumber)
    {
        if (string.IsNullOrWhiteSpace(receiptNumber))
            throw new ArgumentException("Укажите номер чека.", "receiptNumber");
        if (!IsPaid)
        {
            Console.WriteLine($"Заказ #{OrderId}: сначала отметьте оплату.");
            return;
        }

        ReceiptNumber = receiptNumber;
        IsReceiptCreated = true;
    }

    public override void ShowOrderInfo()
    {
        base.ShowOrderInfo();
        ShowDeliveryInfo();
        Console.WriteLine($"Оплата: {PaymentMethod}, чек: {ReceiptNumber}, создан: {IsReceiptCreated}.");
    }
}

class PhysicalOrder : Order
{

    public decimal PackageWeight { get; private set; }
    public int PackageCount { get; private set; }
    public bool IsFragile { get; private set; }

    public void SetPackage(decimal weight, int count)
    {
        if (weight <= 0 || count <= 0)
            throw new ArgumentException("Вес и число упаковок должны быть положительными.");
        PackageWeight = weight;
        PackageCount = count;
    }

    public void MarkAsFragile() { IsFragile = true; }

    public void ShowPackage()
    {
        Console.WriteLine($"Вес: {PackageWeight:F2} кг, упаковок: {PackageCount}, хрупкий: {IsFragile}.");
    }

    public string DeliveryAddress { get; }

    public string CourierName { get; private set; } = "Не назначен";
    public DateTime DeliveryDate { get; set; }
    public bool IsDelivered { get; private set; } = false;

    public string RecipientName { get; private set; } = "Не указан";
    public string RecipientPhone { get; private set; } = "Не указан";
    public string DeliveryComment { get; private set; } = "Нет";

    public PhysicalOrder(int orderId, DateTime creationDate, string deliveryAddress)
        : base(orderId, creationDate)
    {
        if (string.IsNullOrWhiteSpace(deliveryAddress))
            throw new ArgumentException("Адрес доставки не должен быть пустым.", "deliveryAddress");

        DeliveryAddress = deliveryAddress;
        DeliveryDate = creationDate;
    }

    public override bool RemoveItem(Item item)
    {
        bool removed = base.RemoveItem(item);
        if (removed)
            Console.WriteLine($"Физический заказ #{OrderId}: возврат «{item.Name}» зарегистрирован; позиция удалена.");
        else
            Console.WriteLine($"Физический заказ #{OrderId}: позиции «{item.Name}» нет; возврат не оформлен.");

        return removed;
    }

    public void ShowDeliveryAddress()
    {
        Console.WriteLine($"Адрес доставки заказа #{OrderId}: {DeliveryAddress}.");
    }

    public void AssignCourier(string courierName) { CourierName = courierName; }

    public void ShowDeliveryPlan()
    {
        Console.WriteLine($"Доставка #{OrderId}: {DeliveryDate:dd.MM.yyyy}, курьер: {CourierName}, адрес: {DeliveryAddress}.");
    }

    public void MarkAsDelivered()
    {
        IsDelivered = true;
        Status = "Доставлен";
        Console.WriteLine($"Заказ #{OrderId}: доставка завершена: {IsDelivered}.");
    }

    public void SetRecipient(string name, string phone)
    {
        RecipientName = name;
        RecipientPhone = phone;
    }

    public void AssignCourier(string courierName, string deliveryComment)
    {
        AssignCourier(courierName);
        DeliveryComment = deliveryComment;
    }

    public override void ShowOrderInfo()
    {
        base.ShowOrderInfo();
        ShowDeliveryPlan();
        Console.WriteLine($"Получатель: {RecipientName}, телефон: {RecipientPhone}, примечание к доставке: {DeliveryComment}.");
    }
}

class SpecializedOrder : Order
{

    public string CompanyName { get; private set; } = "Не указана";
    public string ContactPerson { get; private set; } = "Не указан";
    public bool IsApproved { get; private set; }

    public void SetCompany(string name, string contact)
    {
        CompanyName = name;
        ContactPerson = contact;
        IsApproved = false;
    }

    public void ApproveConditions() { IsApproved = true; }

    public void ShowCompany()
    {
        Console.WriteLine($"Компания: {CompanyName}, представитель: {ContactPerson}, условия согласованы: {IsApproved}.");
    }

    public string SpecialConditions { get; }
    public decimal DiscountPercent { get; }

    public string ManagerName { get; private set; } = "Не назначен";
    public string GiftMessage { get; private set; } = "Нет";
    public bool IsGift { get; private set; } = false;

    public string GiftWrap { get; private set; } = "Без упаковки";
    public string GiftRecipient { get; private set; } = "Не указан";
    public string GreetingCard { get; private set; } = "Нет";

    public SpecializedOrder(int orderId, DateTime creationDate,
                            string specialConditions, decimal discountPercent)
        : base(orderId, creationDate)
    {
        if (string.IsNullOrWhiteSpace(specialConditions))
            throw new ArgumentException("Укажите специальные условия.", "specialConditions");
        if (discountPercent < 0 || discountPercent > 100)
            throw new ArgumentOutOfRangeException("discountPercent", "Скидка должна быть от 0 до 100 процентов.");

        SpecialConditions = specialConditions;
        DiscountPercent = discountPercent;
    }

    public override decimal CalculateTotal()
    {
        TotalAmount = decimal.Round(base.CalculateTotal() * (1 - DiscountPercent / 100m),
                                    2, MidpointRounding.AwayFromZero);
        return TotalAmount;
    }

    public void ShowSpecialConditions()
    {
        Console.WriteLine($"Условия заказа #{OrderId}: {SpecialConditions}; скидка {DiscountPercent}%.");
    }

    public void AssignManager(string managerName) { ManagerName = managerName; }

    public void MakeGift(string message)
    {
        IsGift = true;
        GiftMessage = message;
    }

    public void ShowGiftInfo()
    {
        Console.WriteLine($"Заказ #{OrderId}: менеджер: {ManagerName}, подарок: {IsGift}, пожелание: {GiftMessage}");
    }

    public void MakeGift(string message, string giftWrap)
    {
        MakeGift(message);
        GiftWrap = giftWrap;
    }

    public void SetGiftRecipient(string name, string greetingCard)
    {
        GiftRecipient = name;
        GreetingCard = greetingCard;
    }

    public override void ShowOrderInfo()
    {
        base.ShowOrderInfo();
        ShowSpecialConditions();
        ShowGiftInfo();
        Console.WriteLine($"Упаковка: {GiftWrap}, получатель подарка: {GiftRecipient}, открытка: {GreetingCard}.");
    }
}

class ExpressOrder : OnlineOrder
{

    public string PickupPoint { get; private set; } = "Не указан";
    public int PriorityLevel { get; private set; } = 1;
    public bool IsCourierCalled { get; private set; }

    public void SetPickupPoint(string point) { PickupPoint = point; }

    public void CallCourier(int priority)
    {
        if (priority < 1 || priority > 3)
            throw new ArgumentOutOfRangeException("priority", "Приоритет должен быть от 1 до 3.");
        PriorityLevel = priority;
        IsCourierCalled = true;
    }

    public void ShowPickup()
    {
        Console.WriteLine($"Место получения: {PickupPoint}, приоритет: {PriorityLevel}, курьер вызван: {IsCourierCalled}.");
    }

    public int DeliveryHours { get; }

    public string CourierName { get; private set; } = "Не назначен";
    public string CourierPhone { get; private set; } = "Не указан";
    public string DeliveryZone { get; private set; } = "Не указана";

    public ExpressOrder(int orderId, DateTime creationDate, string customerEmail,
                        string deliveryMethod, int deliveryHours)
        : base(orderId, creationDate, customerEmail, deliveryMethod)
    {
        if (deliveryHours <= 0)
            throw new ArgumentOutOfRangeException("deliveryHours", "Срок доставки должен быть положительным.");

        DeliveryHours = deliveryHours;
    }

    public void ShowDeadline()
    {
        Console.WriteLine($"Экспресс-заказ #{OrderId}: доставка в течение {DeliveryHours} ч.");
    }

    public void AssignCourier(string name, string phone)
    {
        CourierName = name;
        CourierPhone = phone;
    }

    public void AssignCourier(string name, string phone, string zone)
    {
        AssignCourier(name, phone);
        DeliveryZone = zone;
    }

    public override void ShowOrderInfo()
    {
        base.ShowOrderInfo();
        ShowDeadline();
        Console.WriteLine($"Экспресс-курьер: {CourierName}, телефон: {CourierPhone}, зона: {DeliveryZone}.");
    }
}

class OrderCollection<T> where T : Order
{
    private List<T> orders = new List<T>();

    public int Count { get { return orders.Count; } }

    public void Add(T order)
    {
        if (order == null)
            throw new ArgumentNullException("order");

        orders.Add(order);
    }

    public bool Remove(T order) { return orders.Remove(order); }

    public void ShowAll()
    {
        foreach (T order in orders)
            order.ShowOrderInfo();
    }
}

class OrderService
{
    private readonly INotifiable notifier;

    public OrderService(INotifiable target)
    {
        if (target == null)
            throw new ArgumentNullException("target");
        notifier = target;
    }

    public void SendNotification() { notifier.NotifyCustomer(); }
}

DateTime creationDate = new DateTime(2026, 10, 20);

OnlineOrder onlineOrder1 = new OnlineOrder(101, creationDate, "buyer1@example.com", "курьер");
OnlineOrder onlineOrder2 = new OnlineOrder(102, creationDate, "buyer2@example.com", "пункт выдачи");
PhysicalOrder physicalOrder1 = new PhysicalOrder(201, creationDate, "Учебная улица, дом 1");
PhysicalOrder physicalOrder2 = new PhysicalOrder(202, creationDate, "Учебная улица, дом 2");
SpecializedOrder specializedOrder1 = new SpecializedOrder(301, creationDate, "Скидка постоянному клиенту", 10m);
SpecializedOrder specializedOrder2 = new SpecializedOrder(302, creationDate, "Скидка на услуги", 20m);
ExpressOrder expressOrder = new ExpressOrder(401, new DateTime(2026, 10, 20), "express@example.com", "экспресс-курьер", 6);

Order[] orders = {
    onlineOrder1, onlineOrder2, physicalOrder1, physicalOrder2,
    specializedOrder1, specializedOrder2, expressOrder
};

OrderCollection<Order> allOrders = new OrderCollection<Order>();
foreach (Order order in orders)
{
    order.CustomerName = "Учебный клиент";
    order.Comment = "Позвонить перед доставкой";
    order.ConfirmOrder();
    allOrders.Add(order);
}

Console.WriteLine("Добавление позиций:");
onlineOrder1.AddItem("Мышь", 750m, 2);
Item keyboard = new Item("Клавиатура", 2000m);
onlineOrder1.AddItem(keyboard);
onlineOrder2.AddItem("Флешка", 600m, 3);

Item monitor = new Item("Монитор", 12000m);
physicalOrder1.AddItem(monitor);
Item cable = new Item("Кабель", 500m, 2);
physicalOrder1.AddItem(cable);
physicalOrder2.AddItem("Наушники", 2500m);
specializedOrder1.AddItem("Комплект мебели", 20000m);
specializedOrder2.AddItem("Установка оборудования", 5000m, 2);
expressOrder.AddItem("Веб-камера", 3000m);

Console.WriteLine();
Console.WriteLine("Удаление позиции и информация о возврате:");
physicalOrder1.RemoveItem(monitor);
physicalOrder1.RemoveItem(monitor);

Console.WriteLine();
Console.WriteLine("Перенос позиции между заказами:");
bool transferred = onlineOrder1.TransferItemTo(onlineOrder2, keyboard);
Console.WriteLine($"Позиция «{keyboard.Name}» перенесена из заказа #101 в заказ #102: {transferred}.");

Console.WriteLine();
Console.WriteLine("Изменение количества позиции:");
cable.Quantity = 3;
physicalOrder1.CalculateTotal();
Console.WriteLine($"Кабель: {cable.Quantity} шт.; сумма заказа #201: {physicalOrder1.TotalAmount:F2} руб.");

Console.WriteLine();
Console.WriteLine("Действия с заказами:");
onlineOrder1.ConfirmOrder("+7-900-111-22-33", true);
onlineOrder1.TrackingNumber = "TRK-101";
onlineOrder1.MarkAsPaid("Банковская карта");
onlineOrder1.CreateReceipt("R-101");
onlineOrder1.Track();
onlineOrder1.NotifyCustomer();
onlineOrder2.CancelOrder();
onlineOrder2.Archive();
onlineOrder2.NotifyCustomer();

physicalOrder1.DeliveryDate = new DateTime(2026, 10, 20);
physicalOrder1.AssignCourier("Сергей");
physicalOrder1.MarkAsDelivered();
physicalOrder2.SetRecipient("Ирина", "+7-900-222-33-44");
physicalOrder2.AssignCourier("Олег", "Домофон 23");

specializedOrder1.AssignManager("Анна");
specializedOrder1.MakeGift("С новосельем!", "Крафт");
specializedOrder1.SetGiftRecipient("Елена", "Счастья в новом доме");

expressOrder.TrackingNumber = "EXP-401";
expressOrder.AssignCourier("Виктор", "+7-900-333-44-55", "Центр");
expressOrder.MarkAsPaid("При получении");
expressOrder.CreateReceipt("R-401");
expressOrder.Track();
expressOrder.NotifyCustomer();

Console.WriteLine();
Console.WriteLine("Состав и суммы заказов:");
Console.WriteLine($"Всего заказов: {allOrders.Count}");
allOrders.ShowAll();

OrderCollection<OnlineOrder> onlineOrders = new OrderCollection<OnlineOrder>();
onlineOrders.Add(onlineOrder1);
onlineOrders.Add(onlineOrder2);
onlineOrders.Add(expressOrder);
Console.WriteLine($"Онлайн-заказов: {onlineOrders.Count}");
Console.WriteLine($"Удалён отменённый заказ #102 из онлайн-коллекции: {onlineOrders.Remove(onlineOrder2)}");
Console.WriteLine($"Осталось онлайн-заказов: {onlineOrders.Count}");

Console.WriteLine();
Console.WriteLine("Новые возможности заказа:");
onlineOrder1.SetCity("Тюмень");
onlineOrder1.LeaveReview(5, "Всё доставлено");
onlineOrder1.ShowReview();

onlineOrder1.CreateInvoice("INV-101", "buyer1@example.com");
onlineOrder1.SendInvoice();
onlineOrder1.ShowInvoice();

physicalOrder2.SetPackage(2.5m, 2);
physicalOrder2.MarkAsFragile();
physicalOrder2.ShowPackage();

specializedOrder1.SetCompany("Уют", "Елена");
specializedOrder1.ApproveConditions();
specializedOrder1.ShowCompany();

expressOrder.SetPickupPoint("Склад №1");
expressOrder.CallCourier(2);
expressOrder.ShowPickup();

ITrackable tracking = onlineOrder1;
tracking.Track();
OrderService onlineService = new OrderService(onlineOrder1);
OrderService expressService = new OrderService(expressOrder);
onlineService.SendNotification();
expressService.SendNotification();


Добавление позиций:
Онлайн-заказ #101: добавлено «Мышь»; доставка — курьер.
Онлайн-заказ #101: добавлено «Клавиатура»; доставка — курьер.
Онлайн-заказ #102: добавлено «Флешка»; доставка — пункт выдачи.
Онлайн-заказ #401: добавлено «Веб-камера»; доставка — экспресс-курьер.

Удаление позиции и информация о возврате:
Физический заказ #201: возврат «Монитор» зарегистрирован; позиция удалена.
Физический заказ #201: позиции «Монитор» нет; возврат не оформлен.

Перенос позиции между заказами:
Онлайн-заказ #102: добавлено «Клавиатура»; доставка — пункт выдачи.
Позиция «Клавиатура» перенесена из заказа #101 в заказ #102: True.

Изменение количества позиции:
Кабель: 3 шт.; сумма заказа #201: 1500,00 руб.

Действия с заказами:
Заказ #101: оплата отмечена: True, сумма: 1500,00 руб.
Отслеживание #101: сайт shop.example, номер TRK-101.
Уведомление для buyer1@example.com: заказ #101, статус: Подтверждён.
Уведомление для buyer2@example.com: заказ #102, статус: Отменён.
Заказ #201: доставка завершена: 